# Tiny Transformer: классификация числовых последовательностей

Учебный пример: определить, растёт или падает сигнал датчика внутри окна. Данные синтетические, интернет для их получения не нужен. Архитектура BERT-like: проекция числовых признаков, обучаемые позиции, **двунаправленный Transformer Encoder**, mean pooling и бинарная голова. Это собственная маленькая модель, обучаемая с нуля; здесь нет предобученных весов BERT или masked-language pretraining.

**Ресурсы:** CPU, 1 200 окон × 48 шагов × 3 признака, `d_model=24`, 2 слоя, batch 64, 8 эпох. GPU не требуется. Классы модели импортируются из `src/my_service/model.py`, того же пакета, который использует API. Выполните ячейки сверху вниз. При запуске обновляются артефакт `artifacts/tiny_sequence_transformer_v1.joblib`.

Документация: [TransformerEncoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html), [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [сериализация моделей](https://scikit-learn.org/stable/model_persistence.html).

## Окружение

Выполните `uv sync --locked`, выберите ядро `.venv/bin/python` и выполните ячейки сверху вниз.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import platform
import random
import scipy
import numpy as np
import torch
import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import joblib

SEED = 42
FEATURES = ["sensor_level", "sensor_aux", "sensor_noise"]
SEQ_LEN = 48
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
print({"python": platform.python_version(), "torch": torch.__version__,
       "sklearn": sklearn.__version__, "device": "cpu"})

{'python': '3.11.7', 'torch': '2.14.0', 'sklearn': '1.9.1', 'device': 'cpu'}


## 1. Данные и независимые выборки

Каждая строка набора — отдельное окно, массив имеет форму `(N, T, F)`. Ось `F` строго соответствует `FEATURES`. Класс 1 — положительный тренд `sensor_level`, класс 0 — отрицательный. Амплитуда, смещение, фаза и шум случайны; вспомогательный канал содержит колебания. Около 1% значений отсутствует (`NaN`). Окна генерируются независимо, поэтому случайное разбиение здесь допустимо. Для реальной телеметрии разделяйте по устройству/времени, чтобы соседние окна не попадали в разные выборки.

Масштабирование и заполнение пропусков обучаются только на train. Validation используется только для порога; test — для итоговой оценки. Окна фиксированной длины: padding и маска не нужны.

In [2]:
def make_sequences(n=1200, length=SEQ_LEN, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.arange(n) % 2
    rng.shuffle(y)
    t = np.linspace(-1, 1, length, dtype=np.float32)[None, :]
    phase = rng.uniform(0, 2*np.pi, (n, 1))
    amplitude = rng.uniform(0.6, 1.4, (n, 1))
    offset = rng.normal(0, 0.8, (n, 1))
    trend = (2*y[:, None] - 1) * amplitude * t
    level = offset + trend + 0.25*np.sin(3*np.pi*t + phase)
    level += rng.normal(0, 0.30, (n, length))
    aux = np.sin(2*np.pi*t + phase) + rng.normal(0, 0.25, (n, length))
    noise = rng.normal(0, 1, (n, length))
    X = np.stack([level, aux, noise], axis=-1).astype(np.float32)
    X[rng.random(X.shape) < 0.01] = np.nan
    return X, y.astype(np.int64)

X, y = make_sequences()
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.5, stratify=y_hold, random_state=SEED)
print("train / val / test:", X_train.shape, X_val.shape, X_test.shape)
print("Доля класса 1:", y_train.mean(), "Доля NaN:", np.isnan(X_train).mean())

train / val / test: (840, 48, 3) (180, 48, 3) (180, 48, 3)
Доля класса 1: 0.5 Доля NaN: 0.010168650793650794


## 2. Препроцессинг внутри Pipeline

Заполняем `NaN` средним по каналу, затем стандартизируем. Средние и масштабы вычисляем по всем шагам **только обучающих окон**. Полностью пустой канал обучения и бесконечности считаются ошибками. Контракт проверяет длину окна и число каналов; порядок каналов в безымянном ndarray обязан обеспечить вызывающий код.

In [3]:
from my_service.model import SequencePreprocessor, validate_sequences


## 3. Компактный Encoder и sklearn-обёртка

Без causal mask каждый шаг видит всё окно. Позиционные эмбеддинги помогают различать начало и конец последовательности. `predict_proba` возвращает столбцы `[P(0), P(1)]`; `predict` применяет параметр `threshold`. Веса и обучение остаются на CPU, поэтому бандл не привязан к GPU. Повторный `fit` начинает обучение с нуля.

In [4]:
from my_service.model import TinyTransformerClassifier


In [5]:
pipeline = Pipeline([
    ("preprocessing", SequencePreprocessor()),
    ("model", TinyTransformerClassifier(random_state=SEED)),
])
# Проверяем совместимость с механизмом параметров sklearn до обучения.
assert clone(pipeline).get_params()["model__d_model"] == 24
pipeline.fit(X_train, y_train)
print("Число параметров:", sum(p.numel() for p in pipeline.named_steps["model"].model_.parameters()))

epoch 01/8: loss=0.7004
epoch 02/8: loss=0.6968
epoch 03/8: loss=0.6902
epoch 04/8: loss=0.6656
epoch 05/8: loss=0.3920
epoch 06/8: loss=0.0848
epoch 07/8: loss=0.0332
epoch 08/8: loss=0.0167
Число параметров: 11065


## 4. Порог и оценка

Выбираем порог по F1 на validation из фиксированной сетки. При равенстве F1 предпочитаем ближайший к 0.5. На test порог больше не меняем. Вероятности являются sigmoid-оценками модели, их калибровка здесь не выполняется.

In [6]:
val_p1 = pipeline.predict_proba(X_val)[:, 1]
threshold_grid = np.linspace(0.1, 0.9, 33)
val_scores = np.array([f1_score(y_val, val_p1 >= t, zero_division=0)
                       for t in threshold_grid])
best_candidates = np.flatnonzero(np.isclose(val_scores, val_scores.max()))
best_index = best_candidates[np.argmin(np.abs(threshold_grid[best_candidates] - 0.5))]
threshold = float(threshold_grid[best_index])
pipeline.set_params(model__threshold=threshold)

test_p1 = pipeline.predict_proba(X_test)[:, 1]
test_pred = pipeline.predict(X_test)
test_metrics = {
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "f1": float(f1_score(y_test, test_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, test_p1)),
}
print("threshold:", threshold, "validation F1:", float(val_scores[best_index]))
print("test:", test_metrics)
print("confusion matrix [true rows, predicted columns]:\n", confusion_matrix(y_test, test_pred))

threshold: 0.5 validation F1: 1.0
test: {'accuracy': 1.0, 'f1': 1.0, 'roc_auc': 1.0}
confusion matrix [true rows, predicted columns]:
 [[90  0]
 [ 0 90]]


## 5. Паспорт и joblib-бандл

Сохраняем обученный `Pipeline` и `metadata`: контракт входа, порог, версии и метрики.
Классы импортируются из `my_service.model`. Версия модели после переобучения — 1.1.0.


In [ ]:
metadata = {
    "model_version": "1.1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "binary_sequence_classification",
    "target": "positive_trend_in_sensor_level",
    "model_type": "tiny_bidirectional_transformer_encoder",
    "features": FEATURES.copy(),
    "input_shape": [None, SEQ_LEN, len(FEATURES)],
    "input_dtype": "float32",
    "sequence_length": SEQ_LEN,
    "missing_values": "NaN -> train channel mean; infinity rejected",
    "preprocessing": "per-channel mean imputation and standardization fitted on train",
    "classes": [0, 1],
    "positive_class": 1,
    "threshold": threshold,
    "decision_rule": "predict_proba(X)[:, 1] >= threshold",
    "threshold_selection": "maximum validation F1; ties closest to 0.5",
    "dataset": {"name": "synthetic_independent_sensor_trends", "seed": SEED,
                "n_train": len(y_train), "n_val": len(y_val), "n_test": len(y_test)},
    "model_params": pipeline.named_steps["model"].get_params(),
    "test_metrics": test_metrics,
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "torch": str(torch.__version__), "scikit_learn": sklearn.__version__,
                 "joblib": joblib.__version__, "scipy": scipy.__version__},
    "limitations": ["synthetic data", "fixed-length windows", "uncalibrated probabilities",
                    "feature order must be supplied by caller", "not pretrained BERT"],
}
artifact_path = Path("tiny_sequence_transformer_v1.joblib")
artifact_path.parent.mkdir(parents=True, exist_ok=True)
bundle = {"pipeline": pipeline, "metadata": metadata}
joblib.dump(bundle, artifact_path, compress=3)
print("Сохранён:", artifact_path.resolve())
print("Размер, KiB:", round(artifact_path.stat().st_size / 1024, 1))
print(metadata)


Сохранён: /Users/sabel/Documents/MLPro/artifacts/artifacts/tiny_sequence_transformer_v1.joblib
Размер, KiB: 44.6
{'model_version': '1.1.0', 'created_at_utc': '2026-09-27T11:32:31.756433+00:00', 'task': 'binary_sequence_classification', 'target': 'positive_trend_in_sensor_level', 'model_type': 'tiny_bidirectional_transformer_encoder', 'features': ['sensor_level', 'sensor_aux', 'sensor_noise'], 'input_shape': [None, 48, 3], 'input_dtype': 'float32', 'sequence_length': 48, 'missing_values': 'NaN -> train channel mean; infinity rejected', 'preprocessing': 'per-channel mean imputation and standardization fitted on train', 'classes': [0, 1], 'positive_class': 1, 'threshold': 0.5, 'decision_rule': 'predict_proba(X)[:, 1] >= threshold', 'threshold_selection': 'maximum validation F1; ties closest to 0.5', 'dataset': {'name': 'synthetic_independent_sensor_trends', 'seed': 42, 'n_train': 840, 'n_val': 180, 'n_test': 180}, 'model_params': {'batch_size': 64, 'd_model': 24, 'dim_feedforward': 48, 'd

In [8]:
loaded = joblib.load(artifact_path)
assert set(loaded) == {"pipeline", "metadata"}
restored = loaded["pipeline"]
passport = loaded["metadata"]
assert isinstance(restored, Pipeline)
assert passport["features"] == FEATURES
assert passport["threshold"] == restored.named_steps["model"].threshold
assert np.array_equal(restored.classes_, passport["classes"])

sample = X_test[:16].copy()
sample[0, 0, 0] = np.nan  # пропуски обрабатывает сам Pipeline
before = pipeline.predict_proba(sample)
after = restored.predict_proba(sample)
assert after.shape == (len(sample), 2)
assert np.isfinite(after).all() and ((after >= 0) & (after <= 1)).all()
np.testing.assert_allclose(after.sum(axis=1), 1, atol=1e-6)
np.testing.assert_allclose(after, before, rtol=1e-6, atol=1e-7)
np.testing.assert_array_equal(restored.predict(sample),
                              (after[:, 1] >= passport["threshold"]).astype(int))
np.testing.assert_allclose(restored.predict_proba(sample), after, atol=1e-7)

for invalid in (sample[:, :-1, :], sample[:, :, :-1]):
    try:
        restored.predict_proba(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Некорректная схема должна отклоняться")
print("Smoke-check OK: бандл загружен; вероятности, порог, NaN и проверка формы работают.")
print("Первые P(1):", np.round(after[:5, 1], 4))

Smoke-check OK: бандл загружен; вероятности, порог, NaN и проверка формы работают.
Первые P(1): [0.0111 0.9895 0.0109 0.0111 0.011 ]
